In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
USE CATALOG demo;
USE SCHEMA demo;

In [0]:
%sql
SELECT current_database();

In [0]:
display(dbutils.fs.mounts())

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS f1_demo
MANAGED LOCATION '/mnt/azuredatastorageacc1425/demo'

In [0]:
results_df = spark.read.option("inferSchema",True).json('/mnt/azuredatastorageacc1425/raw/results.json')
display(results_df)

In [0]:
## Writing dataframe as Mangaed Table
results_df.write.format('delta').mode('overwrite').saveAsTable("f1_demo.results_managed")

In [0]:
%sql
select * from f1_demo.results_managed;

In [0]:
results_df.write.format('delta').mode('overwrite').save("/mnt/azuredatastorageacc1425/demo/results_external")

In [0]:
%fs
ls 'abfss://demo@azuredatastorageacc1425.dfs.core.windows.net/'


In [0]:

%sql
-- if LOCATION is provided in SQL then EXTERNAL DELTA table will be created, if not provided then MANAGED DELTA table will be created
CREATE TABLE f1_demo.results_external
USING DELTA
LOCATION 'abfss://demo@azuredatastorageacc1425.dfs.core.windows.net/results_external/'

In [0]:
results_external_df = spark.read.format('delta').load("/mnt/azuredatastorageacc1425/demo/results_external")
display(results_external_df)

In [0]:
results_df.write.format('delta').mode('overwrite').partitionBy("constructorID").saveAsTable("f1_demo.results_partitioned")

In [0]:
%sql
SHOW PARTITIONS f1_demo.results_partitioned

In [0]:
%sql
CREATE TABLE f1_demo.results_managed_partitioned_parquet 
USING PARQUET
LOCATION 'abfss://demo@azuredatastorageacc1425.dfs.core.windows.net/results_parquet_external/'
AS select * from demo.f1_demo.results_partitioned

In [0]:
%sql
-- Update is not allowed in non-Delta tables
UPDATE demo.f1_demo.results_managed_partitioned_parquet
SET points = 11 - position
WHERE position <= 10

In [0]:
%sql
UPDATE demo.f1_demo.results_managed 
SET points = 11 - position
WHERE position <= 10

In [0]:
%sql
-- it is possible to update the external tables
UPDATE demo.f1_demo.results_external 
SET points = 11 - position
WHERE position <= 10

In [0]:
#updating the external delta table using python syntax
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/mnt/azuredatastorageacc1425/demo/results_external")

deltaTable.update("position <= 10", {"points": "11-position"})

In [0]:
%sql
DESCRIBE EXTENDED demo.f1_demo.results_managed

In [0]:
%sql
--drop table demo.f1_demo.results_managed

In [0]:

#updating the managed delta table using python syntax
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "demo.f1_demo.results_managed")

deltaTable.update("position <= 10", {"points": "21 - position"})

In [0]:
%sql
--select current_catalog()
--select current_schema()
--use schema f1_demo
show tables

In [0]:
%sql
-- delete using sql syntax
--DELETE FROM demo.f1_demo.results_managed WHERE position > 10;

In [0]:
#delete records using python syntax
deltaTable.delete("position >10")

In [0]:
%sql
select * from demo.demo.detail_billing_account_en where serviceFamily='Compute'

In [0]:
%sql
select quantity as hours,round(quantity*60) as minutes,round(costInBillingCurrency,2) as price from demo.demo.detail_billing_account_en where serviceFamily='Compute'

In [0]:
%sql
select serviceFamily,sum(costInBillingCurrency)
 from demo.demo.detail_billing_account_en group by 1 order by 2 desc

In [0]:
drivers_day1_df = spark.read.option("inferSchema",True).json('/mnt/azuredatastorageacc1425/raw/drivers.json').filter("driverId <= 10").select('driverId',"dob","name.forename","name.surname")
display(drivers_day1_df)

In [0]:
from pyspark.sql.functions import upper
drivers_day2_df = spark.read.option("inferSchema",True).json('/mnt/azuredatastorageacc1425/raw/drivers.json').filter("driverId BETWEEN 6 AND 15").select('driverId',"dob",upper("name.forename").alias("forename"),upper("name.surname").alias("surname"))
display(drivers_day2_df)

In [0]:
from pyspark.sql.functions import upper
drivers_day3_df = spark.read.option("inferSchema",True).json('/mnt/azuredatastorageacc1425/raw/drivers.json').filter("driverId BETWEEN 1 AND 5 OR driverId BETWEEN 10 AND 20").select('driverId',"dob","name.forename","name.surname")
display(drivers_day3_df)

In [0]:
drivers_day1_df.createOrReplaceTempView("drivers_day1")
drivers_day2_df.createOrReplaceTempView("drivers_day2")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS demo.f1_demo.drivers_merge (
  driverId INT,
  dob DATE,
  forename STRING,
  surname STRING,
  createdDate DATE,
  updatedDate DATE
) USING DELTA

In [0]:
%sql
MERGE INTO demo.f1_demo.drivers_merge tgt
USING drivers_day1 upd
ON tgt.driverId = upd.driverId
WHEN MATCHED THEN UPDATE SET tgt.dob = upd.dob, tgt.forename = upd.forename, tgt.surname = upd.surname, tgt.updatedDate = current_timestamp
WHEN NOT MATCHED THEN INSERT (driverId, dob, forename, surname, createdDate) VALUES (driverId, dob, forename, surname, current_timestamp)

In [0]:
%sql
select * from demo.f1_demo.drivers_merge;

In [0]:
%sql
MERGE INTO demo.f1_demo.drivers_merge tgt
USING drivers_day2 upd
ON tgt.driverId = upd.driverId
WHEN MATCHED THEN UPDATE SET tgt.dob = upd.dob, tgt.forename = upd.forename, tgt.surname = upd.surname, tgt.updatedDate = current_timestamp
WHEN NOT MATCHED THEN INSERT (driverId, dob, forename, surname, createdDate) VALUES (driverId, dob, forename, surname, current_timestamp)

In [0]:
%sql
select * from demo.f1_demo.drivers_merge;

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

deltaTable = DeltaTable.forName(spark, 'demo.f1_demo.drivers_merge')


deltaTable.alias('tgt') \
  .merge(
    drivers_day3_df.alias('upd'),
    'tgt.driverId = upd.driverId'
  ) \
  .whenMatchedUpdate(set =
    {
      "dob": "upd.dob",
      "forename": "upd.forename",
      "surname": "upd.surname",
      "updatedDate": "current_timestamp()",
    }
  ) \
  .whenNotMatchedInsert(values =
    {
      "driverId": "upd.driverId",
      "dob": "upd.dob",
      "forename": "upd.forename",
      "surname": "upd.surname",
      "createdDate": "current_timestamp()",
    }
  ) \
  .execute()

In [0]:
%sql
select * from demo.f1_demo.drivers_merge;

In [0]:
%sql
describe history demo.f1_demo.drivers_merge

In [0]:
%sql
select * from demo.f1_demo.drivers_merge version as of 2;

In [0]:
%sql
select * from demo.f1_demo.drivers_merge timestamp as of '2025-03-17T13:02:25.000+00:00';

In [0]:
df = spark.read.format('delta').option('versionAsOf','5').table('demo.f1_demo.drivers_merge')
display(df)

In [0]:
%sql
VACUUM demo.f1_demo.drivers_merge;

In [0]:
%sql
DESCRIBE HISTORY demo.f1_demo.drivers_merge;

In [0]:
%sql
DELETE FROM demo.f1_demo.drivers_merge WHERE driverId=1

In [0]:
%sql
select * from demo.f1_demo.drivers_merge;

In [0]:
%sql
DESCRIBE HISTORY demo.f1_demo.drivers_merge;

In [0]:
%sql
MERGE INTO demo.f1_demo.drivers_merge tgt
USING demo.f1_demo.drivers_merge VERSION AS OF 7 src
on tgt.driverId = src.driverId
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
select * from demo.f1_demo.drivers_merge;

In [0]:
%sql
describe history demo.f1_demo.drivers_merge;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS demo.f1_demo.drivers_convert_to_delta (
  driverId INT,
  dob DATE,
  forename STRING,
  surname STRING,
  createdDate DATE,
  updatedDate DATE
) USING PARQUET
LOCATION 'abfss://demo@azuredatastorageacc1425.dfs.core.windows.net/drivers_convert_to_delta/';

In [0]:
%sql
INSERT INTO demo.f1_demo.drivers_convert_to_delta
SELECT * FROM demo.f1_demo.drivers_merge;

In [0]:
%sql
--convert parquet table to delta table
CONVERT TO DELTA demo.f1_demo.drivers_convert_to_delta;

In [0]:
df = spark.read.table('demo.f1_demo.drivers_convert_to_delta')
df.write.format('parquet').mode('overwrite').save('abfss://demo@azuredatastorageacc1425.dfs.core.windows.net/drivers_convert_to_delta_new')

In [0]:
%sql
--convert parquet file into delta files
CONVERT TO DELTA parquet.`/mnt/azuredatastorageacc1425/demo/drivers_convert_to_delta_new`